In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/03_dim_organization
# Purpose  : SCD Type 1 via dlt.apply_changes. Logically ordered prior
#            to dim_provider (as providers join against organizations);
#            however, execution sequence is managed automatically by DLT's
#            dependency graph, rendering file numbering purely organizational.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_organization_changes")
def v_organization_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.organizations")
    return df.select(
        F.xxhash64("organization_id").alias("organization_key"),
        "organization_id",
        "organization_name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "revenue",
        "utilization",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_organization")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_organization",
    source="v_organization_changes",
    keys=["organization_id"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)